# LinguoAI – Google-Colab-Version

Diese Ausgabe startet eine passwortgeschützte Weboberfläche. Videos, Stimmreferenzen und Piper-Modelle können direkt hochgeladen oder sicher aus `MyDrive` gelesen werden. NLLB-200 und Faster-Whisper laufen lokal auf GPU oder CPU. Für die Sprachausgabe stehen Edge TTS (online), gTTS/Google Translate TTS (online), Piper (lokales Stimmenmodell) und Chatterbox (lokale Stimmreferenz) zur Wahl.

**Einfachster Start:** Lasse in Zelle 1 die Projektquelle `download_zip` unverändert. Wähle danach **Laufzeit → Alle ausführen**. LinguoAI lädt das geprüfte Projekt-ZIP automatisch von `linguoai.de`; ein manueller Upload ist nur als Alternative nötig.

Optional: **Laufzeit → Laufzeittyp ändern → T4 GPU**. CPU funktioniert, Chatterbox ist darauf aber deutlich langsamer. Wähle das benötigte lokale TTS-Paket in Zelle 3; `none` hält den Basisstart klein und Edge TTS/gTTS funktionieren weiterhin. Nutze eine Stimmreferenz nur mit ausdrücklicher Erlaubnis. Das NLLB-Standardmodell steht unter CC-BY-NC-4.0 und ist ein Forschungsmodell, keine zertifizierte Produktionsübersetzung. `/content` ist flüchtig; Ergebnisse herunterladen oder nach Drive kopieren.

In [ ]:
# @title 1. Projektquelle und Google Drive
import os

PROJECT_SOURCE = "download_zip"  # @param ["download_zip", "upload_zip", "git"]
SOURCE_ZIP_URL = "https://linguoai.de/downloads/LinguoAI-2.0.0-source.zip"  # @param {type:"string"}
GIT_URL = ""  # @param {type:"string"}
GIT_REF = ""  # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
CACHE_MODELS_IN_DRIVE = True  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
if CACHE_MODELS_IN_DRIVE and MOUNT_GOOGLE_DRIVE:
    cache = "/content/drive/MyDrive/LinguoAI/model-cache"
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    os.environ["LINGUOAI_MODEL_CACHE"] = cache
print("Modellcache:", os.environ.get("HF_HOME", "/content (temporär)"))

In [ ]:
# @title 2. Projekt sicher laden
import shutil
import stat
import subprocess
import urllib.request
import zipfile
from pathlib import Path

from google.colab import files

PROJECT_AREA = Path("/content/linguoai-project")
if PROJECT_AREA.exists():
    shutil.rmtree(PROJECT_AREA)
PROJECT_AREA.mkdir(parents=True)


def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        if len(entries) > 10000 or sum(item.file_size for item in entries) > 500 * 1024**2:
            raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
        for info in entries:
            member = Path(info.filename)
            target = (root / member).resolve()
            mode = (info.external_attr >> 16) & 0o170000
            if member.is_absolute() or ".." in member.parts or not target.is_relative_to(root):
                raise ValueError(f"Unsicherer ZIP-Pfad: {info.filename}")
            if mode == stat.S_IFLNK:
                raise ValueError(f"Symlink im ZIP ist nicht erlaubt: {info.filename}")
        archive.extractall(root)


if PROJECT_SOURCE == "git":
    if not GIT_URL.strip():
        raise ValueError("GIT_URL fehlt.")
    repository = PROJECT_AREA / "repo"
    subprocess.run(["git", "clone", "--filter=blob:none", GIT_URL, str(repository)], check=True)
    if GIT_REF.strip():
        subprocess.run(
            ["git", "-C", str(repository), "fetch", "--depth", "1", "origin", GIT_REF],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repository), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
elif PROJECT_SOURCE == "download_zip":
    if not SOURCE_ZIP_URL.startswith("https://"):
        raise ValueError("SOURCE_ZIP_URL muss eine HTTPS-Adresse sein.")
    print("Lade das LinguoAI-Projekt-ZIP:", SOURCE_ZIP_URL)
    archive_path = PROJECT_AREA / "project.zip"
    request = urllib.request.Request(
        SOURCE_ZIP_URL,
        headers={"User-Agent": "LinguoAI-Colab/2.0"},
    )
    downloaded = 0
    with (
        urllib.request.urlopen(request, timeout=120) as response,
        archive_path.open("wb") as handle,
    ):
        if not response.geturl().startswith("https://"):
            raise ValueError("Der Projekt-Download wurde auf eine unsichere Adresse umgeleitet.")
        declared_size = int(response.headers.get("Content-Length", "0") or "0")
        if declared_size > 500 * 1024**2:
            raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
        while chunk := response.read(1024 * 1024):
            downloaded += len(chunk)
            if downloaded > 500 * 1024**2:
                raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
            handle.write(chunk)
    extraction = PROJECT_AREA / "downloaded"
    extraction.mkdir()
    safe_extract_zip(archive_path, extraction)
else:
    print("Bitte das LinguoAI-Projekt als ZIP hochladen.")
    uploaded = files.upload()
    archives = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Bitte genau ein Projekt-ZIP hochladen.")
    archive_path = PROJECT_AREA / "project.zip"
    archive_path.write_bytes(archives[0][1])
    extraction = PROJECT_AREA / "uploaded"
    extraction.mkdir()
    safe_extract_zip(archive_path, extraction)

candidates = [
    p.parent for p in PROJECT_AREA.rglob("pyproject.toml") if (p.parent / "linguoai").is_dir()
]
if len(candidates) != 1:
    raise RuntimeError(f"Projektwurzel nicht eindeutig gefunden: {candidates}")
PROJECT_ROOT = candidates[0]
print("Projekt:", PROJECT_ROOT)

In [ ]:
# @title 3. Abhängigkeiten installieren und Hardware prüfen
import importlib
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

LOCAL_TTS_EXTRA = "none"  # @param ["none", "piper", "chatterbox"]

if "PROJECT_ROOT" not in globals() or not Path(PROJECT_ROOT).exists():
    raise RuntimeError(
        "Das Projekt wurde noch nicht geladen. Bitte zuerst Zelle 1 und Zelle 2 "
        "ausführen oder direkt die letzte Startzelle verwenden."
    )

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

extras = ["colab"]
if LOCAL_TTS_EXTRA == "piper":
    extras.append("piper")
elif LOCAL_TTS_EXTRA == "chatterbox":
    extras.append("voice-clone")
else:
    LOCAL_TTS_EXTRA = "none"

install_target = f"{PROJECT_ROOT}[{','.join(extras)}]"
print("Installiere:", install_target)
if LOCAL_TTS_EXTRA == "chatterbox":
    print("Chatterbox ist ein großes Zusatzpaket; die Installation kann einige Minuten dauern.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", install_target],
    check=True,
)

# Ein editierbar installiertes lokales Projekt ist normalerweise sofort importierbar.
# Der zusätzliche Pfad behebt Colab-Sitzungen, in denen der Import-Cache noch veraltet ist.
project_path = str(Path(PROJECT_ROOT).resolve())
if project_path not in sys.path:
    sys.path.insert(0, project_path)
importlib.invalidate_caches()

if importlib.util.find_spec("linguoai") is None:
    raise RuntimeError(
        "Die Installation wurde ausgeführt, aber das Paket 'linguoai' ist weiterhin nicht "
        "auffindbar. Prüfe, ob das geladene ZIP einen Ordner 'linguoai' und eine "
        "'pyproject.toml' enthält."
    )

if LOCAL_TTS_EXTRA == "piper" and shutil.which("nvidia-smi"):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime", "onnxruntime-gpu"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu>=1.20,<2"],
        check=True,
    )

torch = importlib.import_module("torch")
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name} ({properties.total_memory / 2**30:.1f} GiB)")
else:
    print("Keine GPU erkannt - Whisper, NLLB und lokale TTS verwenden CPU.")
print("Lokales TTS-Paket:", LOCAL_TTS_EXTRA)
print(f"Freier Speicher: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")


## 4. Weboberfläche starten

Die nächste Zelle ist jetzt **selbstreparierend**: Falls `linguoai` noch nicht installiert wurde, lädt sie das Projekt-ZIP, installiert die Colab-Abhängigkeiten und startet anschließend die Weboberfläche. Damit funktioniert sie auch, wenn versehentlich nur die letzte Zelle ausgeführt wurde.

Empfohlen bleibt **Laufzeit → Alle ausführen**, besonders wenn Google Drive oder Piper/Chatterbox verwendet werden sollen. Die Startzelle zeigt Benutzername und ein zufälliges Passwort. In der Oberfläche hat ein direkter Upload Vorrang vor dem jeweiligen Drive-Pfad. Drive-Pfade sind relativ zu `MyDrive`, zum Beispiel `Videos/demo.mp4` oder `LinguoAI/voices/referenz.wav`.


In [ ]:
# @title 4. LinguoAI starten (repariert fehlende Installation automatisch)
import importlib
import importlib.util
import shutil
import stat
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path


def _safe_extract_start_zip(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        if len(entries) > 10000 or sum(item.file_size for item in entries) > 500 * 1024**2:
            raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
        for info in entries:
            member = Path(info.filename)
            target = (root / member).resolve()
            mode = (info.external_attr >> 16) & 0o170000
            if member.is_absolute() or ".." in member.parts or not target.is_relative_to(root):
                raise ValueError(f"Unsicherer ZIP-Pfad: {info.filename}")
            if mode == stat.S_IFLNK:
                raise ValueError(f"Symlink im ZIP ist nicht erlaubt: {info.filename}")
        archive.extractall(root)


def _find_linguoai_root(search_root: Path) -> Path | None:
    candidates = sorted(
        {
            pyproject.parent.resolve()
            for pyproject in search_root.rglob("pyproject.toml")
            if (pyproject.parent / "linguoai").is_dir()
        }
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise RuntimeError(f"Mehrere LinguoAI-Projekte gefunden: {candidates}")
    return None


def _download_project() -> Path:
    source_url = globals().get(
        "SOURCE_ZIP_URL",
        "https://linguoai.de/downloads/LinguoAI-2.0.0-source.zip",
    )
    if not str(source_url).startswith("https://"):
        raise ValueError("SOURCE_ZIP_URL muss eine HTTPS-Adresse sein.")

    project_area = Path("/content/linguoai-project")
    if project_area.exists():
        shutil.rmtree(project_area)
    project_area.mkdir(parents=True)

    archive_path = project_area / "project.zip"
    print("'linguoai' fehlt – Projekt wird automatisch geladen:", source_url)
    request = urllib.request.Request(
        str(source_url),
        headers={"User-Agent": "LinguoAI-Colab/2.0"},
    )
    downloaded = 0
    with urllib.request.urlopen(request, timeout=120) as response, archive_path.open("wb") as handle:
        if not response.geturl().startswith("https://"):
            raise ValueError("Der Projekt-Download wurde auf eine unsichere Adresse umgeleitet.")
        declared_size = int(response.headers.get("Content-Length", "0") or "0")
        if declared_size > 500 * 1024**2:
            raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            downloaded += len(chunk)
            if downloaded > 500 * 1024**2:
                raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
            handle.write(chunk)

    extraction = project_area / "downloaded"
    extraction.mkdir()
    _safe_extract_start_zip(archive_path, extraction)
    root = _find_linguoai_root(extraction)
    if root is None:
        raise RuntimeError(
            "Im heruntergeladenen ZIP wurde kein installierbares LinguoAI-Projekt gefunden. "
            "Erwartet werden 'pyproject.toml' und der Paketordner 'linguoai'."
        )
    return root


# Bereits geladene lokale Quellen zuerst verwenden.
project_root = None
if "PROJECT_ROOT" in globals():
    candidate = Path(PROJECT_ROOT)
    if candidate.exists():
        project_root = candidate.resolve()
if project_root is None:
    existing_area = Path("/content/linguoai-project")
    if existing_area.exists():
        project_root = _find_linguoai_root(existing_area)

# Der Import kann fehlen, wenn nur diese Startzelle ausgeführt wurde.
if importlib.util.find_spec("linguoai") is None:
    if project_root is None:
        project_root = _download_project()

    print("Installiere LinguoAI aus:", project_root)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_root}[colab]"],
        check=True,
    )
    project_path = str(project_root)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    importlib.invalidate_caches()

if importlib.util.find_spec("linguoai") is None:
    raise ModuleNotFoundError(
        "LinguoAI konnte trotz Installation nicht importiert werden. "
        "Öffne 'Laufzeit → Sitzung neu starten' und führe anschließend diese Zelle erneut aus."
    )

from linguoai.colab_ui import launch_colab

launch_colab()
